In [1]:
!pip install -U bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 21.9 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.11.0
    Uninstalling accelerate-1.11.0:
      Successfully uninstalled accelerate-1.11.0


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from transformers.utils import logging
import os
import csv
import time
from tqdm import tqdm

# ---------------------------
# SILENCE WARNINGS
# ---------------------------
logging.set_verbosity_error()

# ---------------------------
# 1. CONFIGURATION
# ---------------------------
# Updated Paths for Mistral
INPUT_FILE_INTERVENTIONS = "/kaggle/input/sufficiencynecessity/esnli_interventions_mistral_updated.csv"
INPUT_FILE_HYPOTHESES    = "/kaggle/input/sufficiencynecessity/esnli_800_with_perturbations.csv"
INPUT_FILE_EXISTING      = "/kaggle/input/sufficiencynecessity/esnli_step3_raw_predictions_mistral.csv"

# New output file containing EVERYTHING
OUTPUT_FILE = "esnli_step3_complete_mistral.csv"

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# ---------------------------
# 2. MODEL SETUP
# ---------------------------
print(f"⏳ Loading {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Mistral-specific padding fix (Critical for batch inference)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=2048,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.1, # Keeps safety penalty
    do_sample=True,
    return_full_text=False
)

print("✅ Model Loaded Successfully.")

# ---------------------------
# 3. PROMPT TEMPLATE
# ---------------------------
SYSTEM_INSTRUCTIONS = """You are an expert NLI reasoning engine. Your task is to classify the logical relationship between a Premise and a Hypothesis.

### INSTRUCTIONS (STRICT):
1. **FUNDAMENTAL RULE**:
   - You must assume the Premise is an indisputable fact in a closed logical world.
   - **Do NOT** evaluate realism. Base your reasoning ONLY on the logical consequences of the premise being true.

2. Determine the correct label:
   - **ENTAILMENT**: The hypothesis MUST be true given the premise.
   - **CONTRADICTION**: The hypothesis CANNOT be true given the premise.
   - **NEUTRAL**: The hypothesis MIGHT be true, but is not guaranteed.

3. Write a concise "Chain of Thought" (1–3 sentences).
   - **Focus on Definitions:** Analyze the concepts abstractly (e.g., "X implies Y").
   - **Deterministic Language:** Avoid hedging ("likely", "possibly").
   - **No Meta-Talk:** Do NOT use phrases like "The premise states...".

### EXAMPLES:

Premise: A soccer player is running across the field.
Hypothesis: A person is moving.
Chain of Thought: The category "soccer player" falls under the category "person", and "running" is a specific type of "movement". Therefore, the premise logically entails the hypothesis.
Final Answer: ENTAILMENT

Premise: A black dog is sleeping on the couch.
Hypothesis: A dog is running outside.
Chain of Thought: The states of "sleeping" and "running" are mutually exclusive; an entity cannot perform both simultaneously. The premise excludes the hypothesis.
Final Answer: CONTRADICTION

Premise: A woman is buying vegetables at the market.
Hypothesis: The woman is preparing dinner.
Chain of Thought: The act of "buying vegetables" does not logically compel "preparing dinner"; acquisition is distinct from preparation. Since the outcome is not guaranteed, the relationship is inconclusive.
Final Answer: NEUTRAL"""

def create_prompt(premise, hypothesis):
    """Generates the formatted string for Mistral."""
    user_content = f"Premise: {premise}\nHypothesis: {hypothesis}\nChain of Thought:"
    
    # Mistral v0.3 supports system prompts via chat template
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": user_content},
    ]
    
    return tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

# ---------------------------
# 4. PREPARE DATA
# ---------------------------
print("📂 Loading files...")
if not os.path.exists(INPUT_FILE_EXISTING):
    print(f"❌ Error: {INPUT_FILE_EXISTING} missing.")
    exit()

df_interv = pd.read_csv(INPUT_FILE_INTERVENTIONS)
df_hypo   = pd.read_csv(INPUT_FILE_HYPOTHESES)
df_exist  = pd.read_csv(INPUT_FILE_EXISTING)

# --- STRICT RENAMING LOGIC ---
# Mapping your specific file names to the internal names the script uses
rename_map = {
    # File Column Name             ->  Internal Script Name
    "premise_original_noise":          "premise_original_sufficiency_control",
    "premise_lexical_noise":           "premise_lexical_sufficiency_control",
    "premise_syntactic_noise":         "premise_syntactic_sufficiency_control",
    "premise_pragmatic_noise":         "premise_pragmatic_sufficiency_control",
    
    # Handling potential typos (just in case)
    "premsie_lexical_noise":           "premise_lexical_sufficiency_control" 
}

print(f"🔄 Renaming columns based on your list...")
df_interv.rename(columns=rename_map, inplace=True)

# Verify Renaming Success
required_cols = [
    "premise_original_sufficiency_control",
    "premise_lexical_sufficiency_control",
    "premise_syntactic_sufficiency_control",
    "premise_pragmatic_sufficiency_control"
]

missing = [c for c in required_cols if c not in df_interv.columns]
if missing:
    print(f"❌ CRITICAL ERROR: After renaming, we are still missing these columns: {missing}")
    print("Columns available in df_interv:", df_interv.columns.tolist())
    exit() 
else:
    print("✅ All required control columns found.")

# --- RENAMING LOGIC 2: Existing Results (Ghost -> Necessity Control) ---
rename_exist_map = {}
variants = ["original", "lexical", "syntactic", "pragmatic"]
for v in variants:
    if f"output_{v}_ghost" in df_exist.columns:
        rename_exist_map[f"output_{v}_ghost"] = f"output_{v}_necessity_control"

if rename_exist_map:
    print(f"🔄 Renaming {len(rename_exist_map)} existing result columns (ghost -> necessity_control)...")
    df_exist.rename(columns=rename_exist_map, inplace=True)

# Merge everything
merged = pd.merge(df_interv, df_hypo[['id', 'Hypothesis', 'perturbation_1_lexical', 'perturbation_2_syntactic', 'perturbation_3_pragmatic']], on="id")
merged = pd.merge(merged, df_exist, on="id", suffixes=('', '_old'))

print(f"🔗 Merged Data. Total rows to process: {len(merged)}")

# ---------------------------
# 5. DEFINE OUTPUT HEADER
# ---------------------------
header = ["id", "gold_label"]
for v in variants:
    header.append(f"coverage_{v}")
    header.append(f"output_{v}_necessity")
    header.append(f"output_{v}_necessity_control")
    header.append(f"output_{v}_sufficiency")
    header.append(f"output_{v}_sufficiency_control")

# ---------------------------
# 6. EXECUTION LOOP
# ---------------------------
if os.path.exists(OUTPUT_FILE):
    existing_done = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(existing_done['id'].unique())
    print(f"🔄 Resuming... {len(processed_ids)} already saved in output.")
else:
    processed_ids = set()
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(header)

print("🚀 Starting Inference on Sufficiency Controls...")

with open(OUTPUT_FILE, 'a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    
    BATCH_SIZE = 4 

    for idx, row in tqdm(merged.iterrows(), total=len(merged), desc="Processing"):
        ex_id = row["id"]
        if ex_id in processed_ids: continue
        
        output_row = [ex_id, row["gold_label"]]
        
        variant_configs = [
            ("original",  row["Hypothesis"]),
            ("lexical",   row["perturbation_1_lexical"]),
            ("syntactic", row["perturbation_2_syntactic"]),
            ("pragmatic", row["perturbation_3_pragmatic"])
        ]
        
        # 1. Prepare Prompts
        batch_prompts = []
        for var_name, hyp_text in variant_configs:
            # Load input text using the now-guaranteed column name
            col_name = f"premise_{var_name}_sufficiency_control"
            input_suff_ctrl = row[col_name]
            batch_prompts.append(create_prompt(input_suff_ctrl, hyp_text))
            
        # 2. Run Batch Inference
        new_outputs = []
        try:
            outputs = pipe(batch_prompts, batch_size=BATCH_SIZE)
            for out in outputs:
                new_outputs.append(out[0]['generated_text'].strip())
        except Exception as e:
            print(f"⚠️ Error on ID {ex_id}: {e}")
            new_outputs = ["ERROR"] * 4
            
        # 3. Construct Final Row
        for i, (var_name, _) in enumerate(variant_configs):
            
            # A. Coverage formatting
            raw_cov = row.get(f"coverage_{var_name}", 0)
            try:
                cov = "{:.4f}".format(float(raw_cov))
            except:
                cov = "0.0000"
            
            # B. Get existing outputs (from merged dataframe)
            out_nec = row.get(f"output_{var_name}_necessity", "")
            out_nec_ctrl = row.get(f"output_{var_name}_necessity_control", "")
            out_suff = row.get(f"output_{var_name}_sufficiency", "")
            
            # C. Get the NEW output
            out_suff_ctrl = new_outputs[i]
            
            output_row.append(cov)
            output_row.append(out_nec)
            output_row.append(out_nec_ctrl)
            output_row.append(out_suff)
            output_row.append(out_suff_ctrl)
            
        # 4. Write to file
        writer.writerow(output_row)
        f.flush()
        os.fsync(f.fileno())

print(f"✅ COMPLETE! All data saved to: {OUTPUT_FILE}")

2026-01-05 13:20:54.716454: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767619254.907245      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767619254.959266      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767619255.401325      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767619255.401362      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767619255.401365      55 computation_placer.cc:177] computation placer alr

⏳ Loading mistralai/Mistral-7B-Instruct-v0.3...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Model Loaded Successfully.
📂 Loading files...
🔄 Renaming columns based on your list...
✅ All required control columns found.
🔄 Renaming 4 existing result columns (ghost -> necessity_control)...
🔗 Merged Data. Total rows to process: 623
🚀 Starting Inference on Sufficiency Controls...


Processing: 100%|██████████| 623/623 [5:55:17<00:00, 34.22s/it]    

✅ COMPLETE! All data saved to: esnli_step3_complete_mistral.csv


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from transformers.utils import logging
import os
import csv
import shutil  # Added for copying the partial file
import time
from tqdm import tqdm

# ---------------------------
# SILENCE WARNINGS
# ---------------------------
logging.set_verbosity_error()

# ---------------------------
# 1. CONFIGURATION
# ---------------------------
# Input Data Files
INPUT_FILE_INTERVENTIONS = "/kaggle/input/sufficiencynecessity/esnli_interventions_mistral_updated.csv"
INPUT_FILE_HYPOTHESES    = "/kaggle/input/sufficiencynecessity/esnli_800_with_perturbations.csv"
INPUT_FILE_EXISTING      = "/kaggle/input/sufficiencynecessity/esnli_step3_raw_predictions_mistral.csv"

# Partial Output File (From your interrupted run)
PARTIAL_FILE = "/kaggle/input/sufficiencynecessity/esnli_step3_complete_mistral.csv"

# Final Output File (Will be created in /kaggle/working/)
OUTPUT_FILE = "esnli_step3_complete_mistral.csv"

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# ---------------------------
# 2. MODEL SETUP
# ---------------------------
print(f"⏳ Loading {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=2048,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.1,
    do_sample=True,
    return_full_text=False
)

print("✅ Model Loaded Successfully.")

# ---------------------------
# 3. PROMPT TEMPLATE
# ---------------------------
SYSTEM_INSTRUCTIONS = """You are an expert NLI reasoning engine. Your task is to classify the logical relationship between a Premise and a Hypothesis.

### INSTRUCTIONS (STRICT):
1. **FUNDAMENTAL RULE**:
   - You must assume the Premise is an indisputable fact in a closed logical world.
   - **Do NOT** evaluate realism. Base your reasoning ONLY on the logical consequences of the premise being true.

2. Determine the correct label:
   - **ENTAILMENT**: The hypothesis MUST be true given the premise.
   - **CONTRADICTION**: The hypothesis CANNOT be true given the premise.
   - **NEUTRAL**: The hypothesis MIGHT be true, but is not guaranteed.

3. Write a concise "Chain of Thought" (1–3 sentences).
   - **Focus on Definitions:** Analyze the concepts abstractly (e.g., "X implies Y").
   - **Deterministic Language:** Avoid hedging ("likely", "possibly").
   - **No Meta-Talk:** Do NOT use phrases like "The premise states...".

### EXAMPLES:

Premise: A soccer player is running across the field.
Hypothesis: A person is moving.
Chain of Thought: The category "soccer player" falls under the category "person", and "running" is a specific type of "movement". Therefore, the premise logically entails the hypothesis.
Final Answer: ENTAILMENT

Premise: A black dog is sleeping on the couch.
Hypothesis: A dog is running outside.
Chain of Thought: The states of "sleeping" and "running" are mutually exclusive; an entity cannot perform both simultaneously. The premise excludes the hypothesis.
Final Answer: CONTRADICTION

Premise: A woman is buying vegetables at the market.
Hypothesis: The woman is preparing dinner.
Chain of Thought: The act of "buying vegetables" does not logically compel "preparing dinner"; acquisition is distinct from preparation. Since the outcome is not guaranteed, the relationship is inconclusive.
Final Answer: NEUTRAL"""

def create_prompt(premise, hypothesis):
    user_content = f"Premise: {premise}\nHypothesis: {hypothesis}\nChain of Thought:"
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": user_content},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# ---------------------------
# 4. PREPARE DATA
# ---------------------------
print("📂 Loading files...")
if not os.path.exists(INPUT_FILE_EXISTING):
    print(f"❌ Error: {INPUT_FILE_EXISTING} missing.")
    exit()

df_interv = pd.read_csv(INPUT_FILE_INTERVENTIONS)
df_hypo   = pd.read_csv(INPUT_FILE_HYPOTHESES)
df_exist  = pd.read_csv(INPUT_FILE_EXISTING)

# --- STRICT RENAMING LOGIC ---
rename_map = {
    "premise_original_noise":          "premise_original_sufficiency_control",
    "premise_lexical_noise":           "premise_lexical_sufficiency_control",
    "premise_syntactic_noise":         "premise_syntactic_sufficiency_control",
    "premise_pragmatic_noise":         "premise_pragmatic_sufficiency_control",
    "premsie_lexical_noise":           "premise_lexical_sufficiency_control" 
}

print(f"🔄 Renaming columns based on your list...")
df_interv.rename(columns=rename_map, inplace=True)

# Verify Renaming Success
required_cols = [
    "premise_original_sufficiency_control",
    "premise_lexical_sufficiency_control",
    "premise_syntactic_sufficiency_control",
    "premise_pragmatic_sufficiency_control"
]

missing = [c for c in required_cols if c not in df_interv.columns]
if missing:
    print(f"❌ CRITICAL ERROR: After renaming, missing columns: {missing}")
    exit() 
else:
    print("✅ All required control columns found.")

# --- RENAMING EXISTING RESULTS (Ghost -> Necessity Control) ---
rename_exist_map = {}
variants = ["original", "lexical", "syntactic", "pragmatic"]
for v in variants:
    if f"output_{v}_ghost" in df_exist.columns:
        rename_exist_map[f"output_{v}_ghost"] = f"output_{v}_necessity_control"

if rename_exist_map:
    print(f"🔄 Renaming {len(rename_exist_map)} existing result columns (ghost -> necessity_control)...")
    df_exist.rename(columns=rename_exist_map, inplace=True)

# Merge everything
merged = pd.merge(df_interv, df_hypo[['id', 'Hypothesis', 'perturbation_1_lexical', 'perturbation_2_syntactic', 'perturbation_3_pragmatic']], on="id")
merged = pd.merge(merged, df_exist, on="id", suffixes=('', '_old'))

print(f"🔗 Merged Data. Total rows to process: {len(merged)}")

# ---------------------------
# 5. DEFINE OUTPUT HEADER
# ---------------------------
header = ["id", "gold_label"]
for v in variants:
    header.append(f"coverage_{v}")
    header.append(f"output_{v}_necessity")
    header.append(f"output_{v}_necessity_control")
    header.append(f"output_{v}_sufficiency")
    header.append(f"output_{v}_sufficiency_control")

# ---------------------------
# 6. RESUME LOGIC (Handling Partial File)
# ---------------------------
processed_ids = set()

# Case A: Check if we have a partial file from a previous session
if os.path.exists(PARTIAL_FILE):
    try:
        print(f"📂 Found partial backup at {PARTIAL_FILE}")
        
        # 1. Copy partial file to working directory
        shutil.copy(PARTIAL_FILE, OUTPUT_FILE)
        print(f"✅ Copied backup to working directory: {OUTPUT_FILE}")
        
        # 2. Read IDs that are already done
        partial_df = pd.read_csv(OUTPUT_FILE)
        processed_ids = set(partial_df['id'].unique())
        print(f"🔄 Resuming from backup... Found {len(processed_ids)} IDs already completed.")
        
    except Exception as e:
        print(f"⚠️ Error processing backup file: {e}")
        # If copy fails, we might start fresh, but usually this works.

# Case B: If no partial file, check if we just started a run in this session
elif os.path.exists(OUTPUT_FILE):
    existing_df = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(existing_df['id'].unique())
    print(f"🔄 Resuming from current session file... {len(processed_ids)} IDs completed.")

# Case C: Start Fresh
else:
    print("✨ No previous partial file found. Starting fresh.")
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(header)

# ---------------------------
# 7. EXECUTION LOOP
# ---------------------------
print("🚀 Starting Inference on Sufficiency Controls...")

with open(OUTPUT_FILE, 'a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    
    BATCH_SIZE = 4 

    for idx, row in tqdm(merged.iterrows(), total=len(merged), desc="Processing"):
        ex_id = row["id"]
        
        # SKIP if already done
        if ex_id in processed_ids: 
            continue
        
        output_row = [ex_id, row["gold_label"]]
        
        variant_configs = [
            ("original",  row["Hypothesis"]),
            ("lexical",   row["perturbation_1_lexical"]),
            ("syntactic", row["perturbation_2_syntactic"]),
            ("pragmatic", row["perturbation_3_pragmatic"])
        ]
        
        # 1. Prepare Prompts
        batch_prompts = []
        for var_name, hyp_text in variant_configs:
            col_name = f"premise_{var_name}_sufficiency_control"
            input_suff_ctrl = row[col_name]
            batch_prompts.append(create_prompt(input_suff_ctrl, hyp_text))
            
        # 2. Run Batch Inference
        new_outputs = []
        try:
            outputs = pipe(batch_prompts, batch_size=BATCH_SIZE)
            for out in outputs:
                new_outputs.append(out[0]['generated_text'].strip())
        except Exception as e:
            print(f"⚠️ Error on ID {ex_id}: {e}")
            new_outputs = ["ERROR"] * 4
            
        # 3. Construct Final Row
        for i, (var_name, _) in enumerate(variant_configs):
            
            # A. Coverage formatting
            raw_cov = row.get(f"coverage_{var_name}", 0)
            try:
                cov = "{:.4f}".format(float(raw_cov))
            except:
                cov = "0.0000"
            
            # B. Get existing outputs (from merged dataframe)
            out_nec = row.get(f"output_{var_name}_necessity", "")
            out_nec_ctrl = row.get(f"output_{var_name}_necessity_control", "")
            out_suff = row.get(f"output_{var_name}_sufficiency", "")
            
            # C. Get the NEW output
            out_suff_ctrl = new_outputs[i]
            
            output_row.append(cov)
            output_row.append(out_nec)
            output_row.append(out_nec_ctrl)
            output_row.append(out_suff)
            output_row.append(out_suff_ctrl)
            
        # 4. Write to file
        writer.writerow(output_row)
        f.flush()
        os.fsync(f.fileno())

print(f"✅ COMPLETE! All data saved to: {OUTPUT_FILE}")

2026-01-06 21:31:20.947418: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767735081.145766      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767735081.200956      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767735081.658301      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767735081.658344      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767735081.658347      55 computation_placer.cc:177] computation placer alr

⏳ Loading mistralai/Mistral-7B-Instruct-v0.3...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

✅ Model Loaded Successfully.
📂 Loading files...
🔄 Renaming columns based on your list...
✅ All required control columns found.
🔄 Renaming 4 existing result columns (ghost -> necessity_control)...
🔗 Merged Data. Total rows to process: 623
📂 Found partial backup at /kaggle/input/sufficiencynecessity/esnli_step3_complete_mistral.csv
✅ Copied backup to working directory: esnli_step3_complete_mistral.csv
🔄 Resuming from backup... Found 566 IDs already completed.
🚀 Starting Inference on Sufficiency Controls...


Processing: 100%|██████████| 623/623 [27:22<00:00,  2.64s/it]

✅ COMPLETE! All data saved to: esnli_step3_complete_mistral.csv


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from transformers.utils import logging
import os
import csv
import time
from tqdm import tqdm

# ---------------------------
# SILENCE WARNINGS
# ---------------------------
logging.set_verbosity_error()

# ---------------------------
# 1. CONFIGURATION
# ---------------------------
# Updated Paths for Qwen
INPUT_FILE = "/kaggle/input/sufficiencynecessity/esnli_interventions_qwen_cleaned.csv"
HYPOTHESIS_FILE = "/kaggle/input/sufficiencynecessity/esnli_800_with_perturbations.csv"
OUTPUT_FILE = "esnli_step3_raw_predictions_qwen.csv"

# Updated Model ID for Qwen 2.5 7B Instruct
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# ---------------------------
# 2. MODEL SETUP
# ---------------------------
print(f"⏳ Loading {MODEL_ID}...")

# 4-bit Quantization Config (Critical for Kaggle T4 GPUs)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Qwen specific padding fix (Critical for batch inference)
# Qwen often maps pad_token_id to None by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=2048,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.1, # Safety penalty to prevent loops
    do_sample=True,
    return_full_text=False
)

print("✅ Model Loaded Successfully.")

# ---------------------------
# 3. PROMPT TEMPLATE
# ---------------------------
SYSTEM_INSTRUCTIONS = """You are an expert NLI reasoning engine. Your task is to classify the logical relationship between a Premise and a Hypothesis.

### INSTRUCTIONS (STRICT):
1. **FUNDAMENTAL RULE**:
   - You must assume the Premise is an indisputable fact in a closed logical world.
   - **Do NOT** evaluate realism. Base your reasoning ONLY on the logical consequences of the premise being true.

2. Determine the correct label:
   - **ENTAILMENT**: The hypothesis MUST be true given the premise.
   - **CONTRADICTION**: The hypothesis CANNOT be true given the premise.
   - **NEUTRAL**: The hypothesis MIGHT be true, but is not guaranteed.

3. Write a concise "Chain of Thought" (1–3 sentences).
   - **Focus on Definitions:** Analyze the concepts abstractly (e.g., "X implies Y").
   - **Deterministic Language:** Avoid hedging ("likely", "possibly").
   - **No Meta-Talk:** Do NOT use phrases like "The premise states...".

### EXAMPLES:

Premise: A soccer player is running across the field.
Hypothesis: A person is moving.
Chain of Thought: The category "soccer player" falls under the category "person", and "running" is a specific type of "movement". Therefore, the premise logically entails the hypothesis.
Final Answer: ENTAILMENT

Premise: A black dog is sleeping on the couch.
Hypothesis: A dog is running outside.
Chain of Thought: The states of "sleeping" and "running" are mutually exclusive; an entity cannot perform both simultaneously. The premise excludes the hypothesis.
Final Answer: CONTRADICTION

Premise: A woman is buying vegetables at the market.
Hypothesis: The woman is preparing dinner.
Chain of Thought: The act of "buying vegetables" does not logically compel "preparing dinner"; acquisition is distinct from preparation. Since the outcome is not guaranteed, the relationship is inconclusive.
Final Answer: NEUTRAL"""

def create_prompt(premise, hypothesis):
    """Generates the formatted string for Qwen."""
    user_content = f"Premise: {premise}\nHypothesis: {hypothesis}\nChain of Thought:"
    
    # Qwen supports system prompts via chat template
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": user_content},
    ]
    
    return tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

# ---------------------------
# 4. DATA LOADING
# ---------------------------
# Verify input files exist before loading
if not os.path.exists(INPUT_FILE):
    print(f"❌ Error: Input file not found at {INPUT_FILE}")
    exit()
if not os.path.exists(HYPOTHESIS_FILE):
    print(f"❌ Error: Hypothesis file not found at {HYPOTHESIS_FILE}")
    exit()

df_interv = pd.read_csv(INPUT_FILE)
df_hypo = pd.read_csv(HYPOTHESIS_FILE)
merged = pd.merge(df_interv, df_hypo[['id', 'Hypothesis', 'perturbation_1_lexical', 'perturbation_2_syntactic', 'perturbation_3_pragmatic', 'gold_label']], on="id")

variants = ["original", "lexical", "syntactic", "pragmatic"]
types = ["necessity", "sufficiency", "ghost"]

header = ["id", "gold_label"]
for v in variants:
    header.append(f"coverage_{v}")
for v in variants:
    for t in types: 
        header.append(f"output_{v}_{t}")

# ---------------------------
# 5. OPTIMIZED INFERENCE LOOP
# ---------------------------
if os.path.exists(OUTPUT_FILE):
    try:
        existing_df = pd.read_csv(OUTPUT_FILE)
        processed_ids = set(existing_df['id'].unique())
        print(f"🔄 Resuming... Found {len(processed_ids)} IDs completed.")
    except:
        processed_ids = set()
else:
    processed_ids = set()
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(header)

with open(OUTPUT_FILE, 'a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    
    BATCH_SIZE = 12 

    for idx, row in tqdm(merged.iterrows(), total=len(merged), desc="Qwen Inference"):
        ex_id = row["id"]
        if ex_id in processed_ids:
            continue
            
        # 1. Collect Metadata
        cov_orig = "{:.4f}".format(float(row["coverage_original"]))
        cov_lex  = "{:.4f}".format(float(row["coverage_lexical"]))
        cov_syn  = "{:.4f}".format(float(row["coverage_syntactic"]))
        cov_prag = "{:.4f}".format(float(row["coverage_pragmatic"]))

        row_results = [ex_id, row["gold_label"], cov_orig, cov_lex, cov_syn, cov_prag]
        
        # 2. Collect ALL Prompts
        batch_prompts = []
        
        variant_data = [
            ("original",  row["Hypothesis"]),
            ("lexical",   row["perturbation_1_lexical"]),
            ("syntactic", row["perturbation_2_syntactic"]),
            ("pragmatic", row["perturbation_3_pragmatic"])
        ]
        
        for var_name, hyp_text in variant_data:
            input_cases = [
                row[f"premise_{var_name}_necessity"],
                row[f"premise_{var_name}_sufficiency"],
                row[f"premise_{var_name}_ghost"]
            ]
            for p_text in input_cases:
                batch_prompts.append(create_prompt(p_text, hyp_text))
        
        # 3. Run Batch Inference
        try:
            outputs = pipe(batch_prompts, batch_size=BATCH_SIZE)
            
            cleaned_outputs = []
            for out in outputs:
                text = out[0]['generated_text'].strip()
                cleaned_outputs.append(text)
                
            row_results.extend(cleaned_outputs)
            
            # 4. Save Row
            writer.writerow(row_results)
            f.flush()
            os.fsync(f.fileno())
            
        except Exception as e:
            print(f"\n⚠️ Error on ID {ex_id}: {e}")
            error_row = row_results + [f"ERROR: {e}"] * 12
            writer.writerow(error_row)

print(f"✅ Step 3 Complete! Saved to {OUTPUT_FILE}")

2026-01-08 20:31:51.146288: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767904311.347070      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767904311.403789      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767904311.882300      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767904311.882334      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767904311.882337      55 computation_placer.cc:177] computation placer alr

⏳ Loading Qwen/Qwen2.5-7B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model Loaded Successfully.


Qwen Inference: 100%|██████████| 627/627 [5:40:28<00:00, 32.58s/it]    

✅ Step 3 Complete! Saved to esnli_step3_raw_predictions_qwen.csv


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from transformers.utils import logging
import os
import csv
import time
from tqdm import tqdm

# ---------------------------
# SILENCE WARNINGS
# ---------------------------
logging.set_verbosity_error()

# ---------------------------
# 1. CONFIGURATION
# ---------------------------
# Updated Paths for Qwen
INPUT_FILE_INTERVENTIONS = "/kaggle/input/sufficiencynecessity/esnli_interventions_qwen_updated_cleaned.csv"
INPUT_FILE_HYPOTHESES    = "/kaggle/input/sufficiencynecessity/esnli_800_with_perturbations.csv"
INPUT_FILE_EXISTING      = "/kaggle/input/sufficiencynecessity/esnli_step3_raw_predictions_qwen.csv"

# New output file containing EVERYTHING
OUTPUT_FILE = "esnli_step3_complete_qwen.csv"

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

# ---------------------------
# 2. MODEL SETUP
# ---------------------------
print(f"⏳ Loading {MODEL_ID}...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Qwen-specific padding fix (Critical for batch inference)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=2048,
    temperature=0.1,
    top_p=0.95,
    repetition_penalty=1.1, # Keeps safety penalty
    do_sample=True,
    return_full_text=False
)

print("✅ Model Loaded Successfully.")

# ---------------------------
# 3. PROMPT TEMPLATE
# ---------------------------
SYSTEM_INSTRUCTIONS = """You are an expert NLI reasoning engine. Your task is to classify the logical relationship between a Premise and a Hypothesis.

### INSTRUCTIONS (STRICT):
1. **FUNDAMENTAL RULE**:
   - You must assume the Premise is an indisputable fact in a closed logical world.
   - **Do NOT** evaluate realism. Base your reasoning ONLY on the logical consequences of the premise being true.

2. Determine the correct label:
   - **ENTAILMENT**: The hypothesis MUST be true given the premise.
   - **CONTRADICTION**: The hypothesis CANNOT be true given the premise.
   - **NEUTRAL**: The hypothesis MIGHT be true, but is not guaranteed.

3. Write a concise "Chain of Thought" (1–3 sentences).
   - **Focus on Definitions:** Analyze the concepts abstractly (e.g., "X implies Y").
   - **Deterministic Language:** Avoid hedging ("likely", "possibly").
   - **No Meta-Talk:** Do NOT use phrases like "The premise states...".

### EXAMPLES:

Premise: A soccer player is running across the field.
Hypothesis: A person is moving.
Chain of Thought: The category "soccer player" falls under the category "person", and "running" is a specific type of "movement". Therefore, the premise logically entails the hypothesis.
Final Answer: ENTAILMENT

Premise: A black dog is sleeping on the couch.
Hypothesis: A dog is running outside.
Chain of Thought: The states of "sleeping" and "running" are mutually exclusive; an entity cannot perform both simultaneously. The premise excludes the hypothesis.
Final Answer: CONTRADICTION

Premise: A woman is buying vegetables at the market.
Hypothesis: The woman is preparing dinner.
Chain of Thought: The act of "buying vegetables" does not logically compel "preparing dinner"; acquisition is distinct from preparation. Since the outcome is not guaranteed, the relationship is inconclusive.
Final Answer: NEUTRAL"""

def create_prompt(premise, hypothesis):
    """Generates the formatted string for Qwen."""
    user_content = f"Premise: {premise}\nHypothesis: {hypothesis}\nChain of Thought:"
    
    # Qwen supports system prompts via chat template
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTIONS},
        {"role": "user", "content": user_content},
    ]
    
    return tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )

# ---------------------------
# 4. PREPARE DATA
# ---------------------------
print("📂 Loading files...")
if not os.path.exists(INPUT_FILE_EXISTING):
    print(f"❌ Error: {INPUT_FILE_EXISTING} missing.")
    exit()

df_interv = pd.read_csv(INPUT_FILE_INTERVENTIONS)
df_hypo   = pd.read_csv(INPUT_FILE_HYPOTHESES)
df_exist  = pd.read_csv(INPUT_FILE_EXISTING)

# --- STRICT RENAMING LOGIC ---
# Mapping your specific file names to the internal names the script uses
rename_map = {
    # File Column Name             ->  Internal Script Name
    "premise_original_noise":          "premise_original_sufficiency_control",
    "premise_lexical_noise":           "premise_lexical_sufficiency_control",
    "premise_syntactic_noise":         "premise_syntactic_sufficiency_control",
    "premise_pragmatic_noise":         "premise_pragmatic_sufficiency_control",
    
    # Handling potential typos (just in case)
    "premsie_lexical_noise":           "premise_lexical_sufficiency_control" 
}

print(f"🔄 Renaming columns based on your list...")
df_interv.rename(columns=rename_map, inplace=True)

# Verify Renaming Success
required_cols = [
    "premise_original_sufficiency_control",
    "premise_lexical_sufficiency_control",
    "premise_syntactic_sufficiency_control",
    "premise_pragmatic_sufficiency_control"
]

missing = [c for c in required_cols if c not in df_interv.columns]
if missing:
    print(f"❌ CRITICAL ERROR: After renaming, we are still missing these columns: {missing}")
    print("Columns available in df_interv:", df_interv.columns.tolist())
    exit() 
else:
    print("✅ All required control columns found.")

# --- RENAMING LOGIC 2: Existing Results (Ghost -> Necessity Control) ---
rename_exist_map = {}
variants = ["original", "lexical", "syntactic", "pragmatic"]
for v in variants:
    if f"output_{v}_ghost" in df_exist.columns:
        rename_exist_map[f"output_{v}_ghost"] = f"output_{v}_necessity_control"

if rename_exist_map:
    print(f"🔄 Renaming {len(rename_exist_map)} existing result columns (ghost -> necessity_control)...")
    df_exist.rename(columns=rename_exist_map, inplace=True)

# Merge everything
merged = pd.merge(df_interv, df_hypo[['id', 'Hypothesis', 'perturbation_1_lexical', 'perturbation_2_syntactic', 'perturbation_3_pragmatic']], on="id")
merged = pd.merge(merged, df_exist, on="id", suffixes=('', '_old'))

print(f"🔗 Merged Data. Total rows to process: {len(merged)}")

# ---------------------------
# 5. DEFINE OUTPUT HEADER
# ---------------------------
header = ["id", "gold_label"]
for v in variants:
    header.append(f"coverage_{v}")
    header.append(f"output_{v}_necessity")
    header.append(f"output_{v}_necessity_control")
    header.append(f"output_{v}_sufficiency")
    header.append(f"output_{v}_sufficiency_control")

# ---------------------------
# 6. EXECUTION LOOP
# ---------------------------
if os.path.exists(OUTPUT_FILE):
    existing_done = pd.read_csv(OUTPUT_FILE)
    processed_ids = set(existing_done['id'].unique())
    print(f"🔄 Resuming... {len(processed_ids)} already saved in output.")
else:
    processed_ids = set()
    with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(header)

print("🚀 Starting Inference on Sufficiency Controls...")

with open(OUTPUT_FILE, 'a', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    
    BATCH_SIZE = 4 

    for idx, row in tqdm(merged.iterrows(), total=len(merged), desc="Qwen Inference"):
        ex_id = row["id"]
        if ex_id in processed_ids: continue
        
        output_row = [ex_id, row["gold_label"]]
        
        variant_configs = [
            ("original",  row["Hypothesis"]),
            ("lexical",   row["perturbation_1_lexical"]),
            ("syntactic", row["perturbation_2_syntactic"]),
            ("pragmatic", row["perturbation_3_pragmatic"])
        ]
        
        # 1. Prepare Prompts
        batch_prompts = []
        for var_name, hyp_text in variant_configs:
            # Load input text using the now-guaranteed column name
            col_name = f"premise_{var_name}_sufficiency_control"
            input_suff_ctrl = row[col_name]
            batch_prompts.append(create_prompt(input_suff_ctrl, hyp_text))
            
        # 2. Run Batch Inference
        new_outputs = []
        try:
            outputs = pipe(batch_prompts, batch_size=BATCH_SIZE)
            for out in outputs:
                new_outputs.append(out[0]['generated_text'].strip())
        except Exception as e:
            print(f"⚠️ Error on ID {ex_id}: {e}")
            new_outputs = ["ERROR"] * 4
            
        # 3. Construct Final Row
        for i, (var_name, _) in enumerate(variant_configs):
            
            # A. Coverage formatting
            raw_cov = row.get(f"coverage_{var_name}", 0)
            try:
                cov = "{:.4f}".format(float(raw_cov))
            except:
                cov = "0.0000"
            
            # B. Get existing outputs (from merged dataframe)
            out_nec = row.get(f"output_{var_name}_necessity", "")
            out_nec_ctrl = row.get(f"output_{var_name}_necessity_control", "")
            out_suff = row.get(f"output_{var_name}_sufficiency", "")
            
            # C. Get the NEW output
            out_suff_ctrl = new_outputs[i]
            
            output_row.append(cov)
            output_row.append(out_nec)
            output_row.append(out_nec_ctrl)
            output_row.append(out_suff)
            output_row.append(out_suff_ctrl)
            
        # 4. Write to file
        writer.writerow(output_row)
        f.flush()
        os.fsync(f.fileno())

print(f"✅ COMPLETE! All data saved to: {OUTPUT_FILE}")

2026-01-09 11:12:43.402395: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767957163.649949      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767957163.717997      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767957164.282510      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767957164.282551      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767957164.282553      55 computation_placer.cc:177] computation placer alr

⏳ Loading Qwen/Qwen2.5-7B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Model Loaded Successfully.
📂 Loading files...
🔄 Renaming columns based on your list...
✅ All required control columns found.
🔄 Renaming 4 existing result columns (ghost -> necessity_control)...
🔗 Merged Data. Total rows to process: 627
🚀 Starting Inference on Sufficiency Controls...


Qwen Inference: 100%|██████████| 627/627 [3:49:01<00:00, 21.92s/it]    

✅ COMPLETE! All data saved to: esnli_step3_complete_qwen.csv
